
# Benchmark 3 (v2): Kagome paper-order diagnosis with `unclassified` outlet

This notebook tests the updated `kagome_order_diagnosis_v2.py`.

Goals:
1. Verify that the five paper orders can still be recognized when the synthetic kernel matches their templates.
2. Verify that a physically meaningful **out-of-paper** order is **not forced** into one of the five labels.
3. Confirm that the diagnoser returns:
   - `paper_label = "unclassified"`
   - `recognition_status = "novel_candidate"`

The extra test case below uses a **pp singlet d-wave superconducting mode**.
It is a sensible instability, but it is *not* one of the five phases discussed in the PRL paper
(which only has `f-SC` in the superconducting sector).


In [1]:

import numpy as np
from dataclasses import dataclass
from pprint import pprint

from form_factor import OrderRecognizer
from kagome_order_diagnosis import KagomeOrderDiagnoser


In [2]:

@dataclass
class Patch:
    k_cart: np.ndarray
    eigvec: np.ndarray

@dataclass
class PatchSet:
    patches: list

@dataclass
class Kernel:
    name: str
    Q: np.ndarray
    matrix: np.ndarray
    row_partner_patches: np.ndarray
    col_partner_patches: np.ndarray

    def eig(self, sort_by="abs"):
        evals, evecs = np.linalg.eigh(self.matrix)
        if sort_by == "abs":
            idx = np.argsort(np.abs(evals))[::-1]
        elif sort_by == "real":
            idx = np.argsort(evals.real)[::-1]
        else:
            idx = np.argsort(evals)[::-1]
        return evals[idx], evecs[:, idx]


In [3]:

def kagome_like_patchset(npatch=24):
    thetas = np.linspace(0, 2*np.pi, npatch, endpoint=False)
    patches = []
    for th in thetas:
        k = np.array([np.cos(th), np.sin(th)], dtype=float)
        # simple smooth 3-sublattice weights
        u = np.array([
            1.0,
            np.exp(1j * th),
            np.exp(-1j * th),
        ], dtype=complex)
        u = u / np.linalg.norm(u)
        patches.append(Patch(k_cart=k, eigvec=u))
    return PatchSet(patches), thetas

patchset, thetas = kagome_like_patchset(24)
recognizer = OrderRecognizer(patchset)
diagnoser = KagomeOrderDiagnoser(patchset, recognizer=recognizer, min_paper_template_score=0.12)


In [4]:

def projector_kernel(name, Q, vecs, partner_map):
    V = np.column_stack([np.asarray(v, dtype=complex) / np.linalg.norm(v) for v in vecs])
    mat = V @ V.conjugate().T
    partner = np.asarray(partner_map, dtype=int)
    return Kernel(
        name=name,
        Q=np.asarray(Q, dtype=float),
        matrix=mat,
        row_partner_patches=partner.copy(),
        col_partner_patches=partner.copy(),
    )

npatch = len(thetas)
identity_partner = np.arange(npatch, dtype=int)
pp_partner = (np.arange(npatch, dtype=int) + npatch//2) % npatch
Q1 = np.array([-0.5, -np.sqrt(3)/2])


## Build six synthetic cases

In [5]:

# 1) PI : Q=0, ph, d-wave doublet
v_d1 = np.cos(2*thetas)
v_d2 = np.sin(2*thetas)
ker_PI = projector_kernel("ph_charge", np.zeros(2), [v_d1, v_d2], identity_partner)

# 2) cBO : finite-Q, ph charge-like off-diagonal BO
v_bo = np.sin(np.cos(thetas)*Q1[0] + np.sin(thetas)*Q1[1])
ker_cBO = projector_kernel("ph_charge", Q1, [v_bo], identity_partner)

# 3) sBO : finite-Q, ph spin-like off-diagonal BO
ker_sBO = projector_kernel("ph_spin", Q1, [v_bo], identity_partner)

# 4) f-SC : pp triplet odd mode
v_f = np.cos(3*thetas)
ker_fSC = projector_kernel("pp_triplet_sz0", np.zeros(2), [v_f], pp_partner)

# 5) FM : Q=0 spin-like constant ph mode
v_s = np.ones(npatch)
ker_FM = projector_kernel("ph_spin", np.zeros(2), [v_s], identity_partner)

# 6) NEW / out-of-paper case: pp singlet d-wave SC
# This is a sensible superconducting order, but not one of the five paper labels.
ker_new = projector_kernel("pp_singlet", np.zeros(2), [v_d1, v_d2], pp_partner)

kernels = {
    "PI": ker_PI,
    "cBO": ker_cBO,
    "sBO": ker_sBO,
    "fSC": ker_fSC,
    "FM": ker_FM,
    "new_pp_dwave": ker_new,
}


In [6]:

results = {}
for name, ker in kernels.items():
    res = diagnoser.diagnose_kernel(ker)
    results[name] = res
    print("="*80)
    print(name)
    pprint(res.summary_dict())


PI
{'Q': [0.0, 0.0],
 'channel_sector': 'ph',
 'coarse_label': 'd',
 'coarse_score': 1.0000000000000004,
 'degeneracy': 2,
 'dominant_pair': (0, 0),
 'inter_sublattice_weight': 0.6666666666666666,
 'kernel': 'ph_charge',
 'notes': ['Q≈0 particle-hole mode: candidate Pomeranchuk / nematic sector.',
           'Leading space is at least two-dimensional; treat it as an irrep '
           'subspace, not one fixed basis vector.',
           'Particle-hole patch harmonics can diagnose Q and rough lattice '
           'symmetry, but real-space bond-order identification still needs a '
           'separate bond/sublattice reconstruction layer.',
           'Q≈0 particle-hole d-wave-like mode with diagonal / same-sublattice '
           'dominance.',
           'Treat as twofold-degenerate d-wave PI subspace rather than one '
           'fixed basis vector.',
           'Dominant reconstructed sublattice pair: A-A.'],
 'paper_label': 'PI',
 'paper_score': 0.33026732487248656,
 'recognition_stat

## Minimal checks

In [7]:

assert results["PI"].paper_label == "PI"
assert results["cBO"].paper_label == "cBO"
assert results["sBO"].paper_label == "sBO"
assert results["fSC"].paper_label == "f-SC"
assert results["FM"].paper_label == "FM"

assert results["new_pp_dwave"].paper_label == "unclassified"
assert results["new_pp_dwave"].recognition_status == "novel_candidate"

print("All checks passed.")


AssertionError: 


## What the new case demonstrates

The `new_pp_dwave` kernel is intentionally **physical**:
- particle-particle channel
- singlet
- d-wave-like angular structure

But it is **not** one of the five PRL paper orders.

So the updated diagnoser should **not** force it into:
- `PI`
- `cBO`
- `sBO`
- `f-SC`
- `FM`

Instead, it should keep:
- `coarse_label` from the generic recognizer
- `paper_label = "unclassified"`
- `recognition_status = "novel_candidate"`

That is the desired behavior for future model exploration.
